# Experiment 1: Preset Throughput Sweep

Exports throughput figures per setting (`silo`, `wifi`, `lte`) and reliability tables (delay/jitter/loss).

In [2]:
from pathlib import Path
import csv
import json
import math
from typing import Any
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (6, 4),      # width, height in inches
    "font.size": 14,               # base font size
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.dpi": 300,             # high resolution for paper
})
import numpy as np
import pandas as pd
def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'orchestrator.py').exists() and (candidate / 'experiments').is_dir():
            return candidate
    return None
ROOT = find_repo_root(Path.cwd())
if ROOT is None:
    raise RuntimeError(f'Could not locate fl-net-sim repo root from current working directory: {Path.cwd()}')
EXP_ID = 'exp1_presets'
EXP_DIR = ROOT / 'experiments' / EXP_ID
MANIFEST_PATH = EXP_DIR / 'outputs' / 'last_run_manifest.json'
SOURCE_EXPORTS_DIR = EXP_DIR / 'outputs' / 'source_exports'
TABLES_DIR = EXP_DIR / 'outputs' / 'tables'
FIGURES_DIR = EXP_DIR / 'outputs' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SETTING_ORDER = ['silo', 'wifi', 'lte']
PRESET_ORDER = ['very_weak', 'weak', 'basic', 'strong', 'very_strong']
# Limit postprocessing to a subset if needed (e.g. ['wifi']).
# Keep as None to process all settings.
ACTIVE_SETTINGS_OVERRIDE: list[str] | None = None
if ACTIVE_SETTINGS_OVERRIDE is None:
    ACTIVE_SETTINGS = list(SETTING_ORDER)
else:
    requested = {str(s).strip() for s in ACTIVE_SETTINGS_OVERRIDE}
    ACTIVE_SETTINGS = [s for s in SETTING_ORDER if s in requested]
    if not ACTIVE_SETTINGS:
        raise ValueError(f'No valid settings in ACTIVE_SETTINGS_OVERRIDE={ACTIVE_SETTINGS_OVERRIDE}. Expected any of {SETTING_ORDER}.')


In [3]:
# Folder-first discovery: scan outputs/source_exports directly.
# Manifest is optional and used only to enrich metadata fields.
manifest_runs_by_id: dict[str, dict[str, Any]] = {}
if MANIFEST_PATH.exists():
    try:
        manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        for item in manifest.get('runs', []):
            cid = str(item.get('config_id', '')).strip()
            if cid:
                manifest_runs_by_id[cid] = item
        print(f'Loaded manifest metadata from {MANIFEST_PATH} (runs={len(manifest_runs_by_id)})')
    except Exception as exc:
        print(f'Warning: failed to parse manifest {MANIFEST_PATH}: {exc}')
if not SOURCE_EXPORTS_DIR.exists():
    raise FileNotFoundError(f'Missing source exports directory: {SOURCE_EXPORTS_DIR}. Run the experiment first.')
runs: list[dict[str, Any]] = []
for run_dir in sorted([p for p in SOURCE_EXPORTS_DIR.iterdir() if p.is_dir()]):
    config_id = run_dir.name
    if '_' not in config_id:
        continue
    setting, preset = config_id.split('_', 1)
    if setting not in SETTING_ORDER or preset not in PRESET_ORDER:
        continue
    if setting not in ACTIVE_SETTINGS:
        continue
    item: dict[str, Any] = {
        'config_id': config_id,
        'setting': setting,
        'preset': preset,
        'exports_dir': str(run_dir),
        'copied_requested_rounds_csv': str(run_dir / 'requested_rounds.csv') if (run_dir / 'requested_rounds.csv').exists() else None,
        'copied_requested_rounds_json': str(run_dir / 'requested_rounds.json') if (run_dir / 'requested_rounds.json').exists() else None,
        'copied_all_rounds_csv': str(run_dir / 'all_rounds.csv') if (run_dir / 'all_rounds.csv').exists() else None,
        'copied_all_rounds_json': str(run_dir / 'all_rounds.json') if (run_dir / 'all_rounds.json').exists() else None,
        'copied_round_index_csv': str(run_dir / 'round_index.csv') if (run_dir / 'round_index.csv').exists() else None,
        'copied_round_states_csv': str(run_dir / 'round_states.csv') if (run_dir / 'round_states.csv').exists() else None,
        'exec_time_s': float('nan'),
        'rounds_requested': 0,
        'planned_rounds': 0,
        'cached_rounds': 0,
        'deduped_rounds': 0,
    }
    meta = manifest_runs_by_id.get(config_id)
    if isinstance(meta, dict):
        for key in ('exec_time_s', 'rounds_requested', 'planned_rounds', 'cached_rounds', 'deduped_rounds'):
            if key in meta:
                item[key] = meta[key]
    runs.append(item)
if not runs:
    raise ValueError(f'No matching source exports found for ACTIVE_SETTINGS={ACTIVE_SETTINGS} under {SOURCE_EXPORTS_DIR}')
runs_df = pd.DataFrame(runs)
runs_df[['config_id', 'setting', 'preset', 'exec_time_s', 'rounds_requested', 'planned_rounds', 'cached_rounds', 'deduped_rounds']]


Loaded manifest metadata from /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/last_run_manifest.json (runs=5)


,config_id,setting,preset,exec_time_s,rounds_requested,planned_rounds,cached_rounds,deduped_rounds
0,lte_basic,lte,basic,NaN,0,0,0,0
1,lte_strong,lte,strong,NaN,0,0,0,0
2,lte_very_strong,lte,very_strong,NaN,0,0,0,0
3,lte_very_weak,lte,very_weak,NaN,0,0,0,0
4,lte_weak,lte,weak,NaN,0,0,0,0
5,silo_basic,silo,basic,NaN,0,0,0,0
6,silo_strong,silo,strong,NaN,0,0,0,0
7,silo_very_strong,silo,very_strong,NaN,0,0,0,0
8,silo_very_weak,silo,very_weak,NaN,0,0,0,0
9,silo_weak,silo,weak,NaN,0,0,0,0


In [4]:
def load_round_table(path: Path) -> pd.DataFrame:
    text = path.read_text(encoding='utf-8')
    section = text.split('\n\n', 1)[0].strip()
    rows = list(csv.DictReader(section.splitlines()))
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    for col in frame.columns:
        converted = pd.to_numeric(frame[col], errors='coerce')
        if converted.notna().any():
            frame[col] = converted
    return frame

def pick_csv_path(item: dict[str, Any]) -> Path | None:
    candidate = (
        item.get('copied_requested_rounds_csv')
        or item.get('requested_rounds_csv')
        or item.get('copied_all_rounds_csv')
        or item.get('all_rounds_csv')
    )
    if not candidate:
        return None
    p = Path(candidate)
    return p if p.exists() else None

def p95(series: pd.Series) -> float:
    clean = pd.to_numeric(series, errors='coerce').dropna()
    return float(clean.quantile(0.95)) if not clean.empty else float('nan')

records: list[dict[str, Any]] = []
per_round_frames: list[pd.DataFrame] = []

for item in runs:
    config_id = item.get('config_id')
    setting = str(item.get('setting', '')).strip()
    preset = str(item.get('preset', '')).strip()

    csv_path = pick_csv_path(item)
    if csv_path is None:
        continue

    frame = load_round_table(csv_path)
    if frame.empty:
        continue

    frame['config_id'] = config_id
    frame['setting'] = setting
    frame['preset'] = preset
    per_round_frames.append(frame)

    def mean_col(name: str) -> float:
        if name not in frame.columns:
            return float('nan')
        clean = pd.to_numeric(frame[name], errors='coerce').dropna()
        return float(clean.mean()) if not clean.empty else float('nan')

    def p95_col(name: str) -> float:
        if name not in frame.columns:
            return float('nan')
        return p95(frame[name])

    records.append({
        'config_id': config_id,
        'setting': setting,
        'preset': preset,
        'exec_time_s': float(item.get('exec_time_s', float('nan'))),
        'rounds_requested': int(item.get('rounds_requested', 0)),
        'planned_rounds': int(item.get('planned_rounds', 0)),
        'cached_rounds': int(item.get('cached_rounds', 0)),
        'deduped_rounds': int(item.get('deduped_rounds', 0)),
        'num_round_rows': int(len(frame)),
        'mean_dl_throughput_mbps': mean_col('agg_dl_goodput_mbps'),
        'mean_ul_throughput_mbps': mean_col('agg_ul_goodput_mbps'),
        'p95_dl_throughput_mbps': p95_col('agg_dl_goodput_mbps'),
        'p95_ul_throughput_mbps': p95_col('agg_ul_goodput_mbps'),
        'mean_delay_ms': mean_col('mean_delay_ms'),
        'p95_delay_ms': p95_col('mean_delay_ms'),
        'mean_jitter_ms': mean_col('mean_jitter_ms'),
        'p95_jitter_ms': p95_col('mean_jitter_ms'),
        'mean_packet_loss_pct': mean_col('packet_loss_pct'),
        'p95_packet_loss_pct': p95_col('packet_loss_pct'),
    })

summary_df = pd.DataFrame(records)
if summary_df.empty:
    raise ValueError('No round-level metrics found. Check outputs/source_exports and exported CSV files.')

summary_df['setting'] = pd.Categorical(summary_df['setting'], categories=SETTING_ORDER, ordered=True)
summary_df['preset'] = pd.Categorical(summary_df['preset'], categories=PRESET_ORDER, ordered=True)
summary_df = summary_df.sort_values(['setting', 'preset']).reset_index(drop=True)

throughput_summary = summary_df[[
    'config_id',
    'setting',
    'preset',
    'mean_dl_throughput_mbps',
    'mean_ul_throughput_mbps',
    'p95_dl_throughput_mbps',
    'p95_ul_throughput_mbps',
    'num_round_rows',
    'deduped_rounds',
]].copy()

reliability_summary = summary_df[[
    'config_id',
    'setting',
    'preset',
    'mean_delay_ms',
    'p95_delay_ms',
    'mean_jitter_ms',
    'p95_jitter_ms',
    'mean_packet_loss_pct',
    'p95_packet_loss_pct',
    'num_round_rows',
]].copy()

all_round_metrics = pd.concat(per_round_frames, ignore_index=True) if per_round_frames else pd.DataFrame()
throughput_summary


,config_id,setting,preset,mean_dl_throughput_mbps,mean_ul_throughput_mbps,p95_dl_throughput_mbps,p95_ul_throughput_mbps,num_round_rows,deduped_rounds
0,silo_very_weak,silo,very_weak,11.006304,5.307809,13.044033,5.697531,5,0
1,silo_weak,silo,weak,28.504827,13.472417,34.984294,15.725218,5,0
2,silo_basic,silo,basic,114.747215,41.733391,160.165079,49.722652,5,0
3,silo_strong,silo,strong,304.046779,109.475245,332.012837,113.240024,5,0
4,silo_very_strong,silo,very_strong,456.748778,155.581341,469.812440,165.054982,5,0
5,wifi_very_weak,wifi,very_weak,6.048862,3.651773,6.328433,4.015121,5,0
6,wifi_weak,wifi,weak,6.404890,4.265510,7.400386,5.516550,5,0
7,wifi_basic,wifi,basic,22.606669,16.230933,28.494242,20.331250,5,0
8,wifi_strong,wifi,strong,23.335532,16.546071,27.084462,19.597949,5,0
9,wifi_very_strong,wifi,very_strong,22.629512,15.729987,27.789027,17.422168,5,0


In [5]:
def to_markdown_no_deps(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except Exception:
        safe = df.fillna('')
        cols = [str(c) for c in safe.columns]
        header = '| ' + ' | '.join(cols) + ' |'
        sep = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
        rows = [
            '| ' + ' | '.join(str(v).replace('\n', ' ').replace('|', '\\|') for v in row) + ' |'
            for row in safe.itertuples(index=False, name=None)
        ]
        return '\n'.join([header, sep, *rows])

settings_tag = 'all' if ACTIVE_SETTINGS == SETTING_ORDER else '_'.join(ACTIVE_SETTINGS)
file_prefix = 'exp1' if settings_tag == 'all' else f'exp1_{settings_tag}'

throughput_csv = TABLES_DIR / f'{file_prefix}_throughput_summary.csv'
throughput_md = TABLES_DIR / f'{file_prefix}_throughput_summary.md'
reliability_csv = TABLES_DIR / f'{file_prefix}_reliability_summary.csv'
reliability_md = TABLES_DIR / f'{file_prefix}_reliability_summary.md'
per_round_csv = TABLES_DIR / f'{file_prefix}_per_round_metrics.csv'

throughput_summary.to_csv(throughput_csv, index=False)
throughput_md.write_text(to_markdown_no_deps(throughput_summary) + '\n', encoding='utf-8')
reliability_summary.to_csv(reliability_csv, index=False)
reliability_md.write_text(to_markdown_no_deps(reliability_summary) + '\n', encoding='utf-8')

if not all_round_metrics.empty:
    export_cols = [c for c in [
        'config_id', 'setting', 'preset', 'round',
        'agg_dl_goodput_mbps', 'agg_ul_goodput_mbps',
        'mean_delay_ms', 'mean_jitter_ms', 'packet_loss_pct'
    ] if c in all_round_metrics.columns]
    all_round_metrics[export_cols].to_csv(per_round_csv, index=False)

print(f'wrote {throughput_csv}')
print(f'wrote {throughput_md}')
print(f'wrote {reliability_csv}')
print(f'wrote {reliability_md}')
if not all_round_metrics.empty:
    print(f'wrote {per_round_csv}')

reliability_summary


wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/tables/exp1_throughput_summary.csv
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/tables/exp1_throughput_summary.md
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/tables/exp1_reliability_summary.csv
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/tables/exp1_reliability_summary.md
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/tables/exp1_per_round_metrics.csv


,config_id,setting,preset,mean_delay_ms,p95_delay_ms,mean_jitter_ms,p95_jitter_ms,mean_packet_loss_pct,p95_packet_loss_pct,num_round_rows
0,silo_very_weak,silo,very_weak,40.024450,43.177274,0.111961,0.117434,0.172350,0.197036,5
1,silo_weak,silo,weak,30.793035,33.286250,0.053726,0.055521,0.117270,0.125943,5
2,silo_basic,silo,basic,20.094330,21.619360,0.025790,0.026650,0.043883,0.050781,5
3,silo_strong,silo,strong,14.217903,14.631611,0.012251,0.012480,0.017661,0.030414,5
4,silo_very_strong,silo,very_strong,8.372873,8.903369,0.007870,0.008634,0.002251,0.004221,5
5,wifi_very_weak,wifi,very_weak,82.686143,87.852645,1.998916,2.074878,0.941143,1.156267,5
6,wifi_weak,wifi,weak,78.398729,88.026268,1.807744,2.162000,0.785902,1.178910,5
7,wifi_basic,wifi,basic,34.849915,42.449982,0.438785,0.628299,0.251976,0.348306,5
8,wifi_strong,wifi,strong,33.867811,39.671619,0.388784,0.489720,0.291434,0.376619,5
9,wifi_very_strong,wifi,very_strong,31.777509,36.562557,0.421385,0.474097,0.426002,0.573497,5


In [6]:
print(ACTIVE_SETTINGS)
for setting in ACTIVE_SETTINGS:
    group = throughput_summary[throughput_summary['setting'] == setting].copy()
    if group.empty:
        continue

    group['preset'] = pd.Categorical(group['preset'], categories=PRESET_ORDER, ordered=True)
    group = group.sort_values('preset').reset_index(drop=True)

    x = np.arange(len(group))
    width = 0.38

    plt.figure(figsize=(10, 5))
    plt.bar(x - width / 2, group['mean_dl_throughput_mbps'], width=width, label='DL throughput (Mbps)')
    plt.bar(x + width / 2, group['mean_ul_throughput_mbps'], width=width, label='UL throughput (Mbps)')

    
    label_setting = 'Cross-Device' if setting == 'lte' else 'Cross-Silo' if setting == 'silo' else setting.capitalize()
    print(f'label_setting for setting="{setting}": {label_setting}')
    plt.title(f'Connectivity Setting ({label_setting}): Throughput by Preset')
    plt.xlabel('Preset')
    plt.ylabel('Mean Throughput (Mbps)')
    plt.xticks(x, group['preset'].astype(str), rotation=20, ha='right')
    plt.legend()
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()

    fig_path = FIGURES_DIR / f'exp1_{setting}_throughput_by_preset.png'
    plt.savefig(fig_path, dpi=180)
    plt.close()
    print(f'wrote {fig_path}')


['silo', 'wifi', 'lte']
label_setting for setting="silo": Cross-Silo
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/figures/exp1_silo_throughput_by_preset.png
label_setting for setting="wifi": Wifi
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/figures/exp1_wifi_throughput_by_preset.png
label_setting for setting="lte": Cross-Device
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp1_presets/outputs/figures/exp1_lte_throughput_by_preset.png
